# Person 3 - Gradient Boosting (T-2h)

This notebook trains **only on development data** (`train_T2h.csv` and `validation_T2h.csv`). It never reads the December test file. It uses the shared four expanding chronological folds, ranks models by AP / PR-AUC, and writes the Person 4 handoff files.

Run the cells in order. A full run trains HistGradientBoosting, XGBoost, LightGBM, and CatBoost for 4 folds x 3 seeds, so leave the kernel running. Start with `RUN_MODE = 'smoke'` if you want to verify installation and the data path first.

In [ ]:
# Run once if these packages are not already installed. Restart the kernel after installation.
%pip install -q 'scikit-learn>=1.5' xgboost lightgbm catboost pyarrow joblib

In [ ]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

PROJECT_ROOT = Path('..') if Path('../final_csv').exists() else Path('.')
DATA_DIR = PROJECT_ROOT / 'final_csv'
OUT_DIR = PROJECT_ROOT / 'person3_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_MODE = 'smoke'  # change to 'full' after the smoke run succeeds
MODELS = ['hist_gradient_boosting', 'xgboost', 'lightgbm', 'catboost']
SEEDS = (11, 23, 42)
N_JOBS = -1

# The smoke run is a path/dependency check, not a reportable model result.
MAX_TRAIN_ROWS = 75_000 if RUN_MODE == 'smoke' else 0
MODELS_TO_RUN = ['hist_gradient_boosting'] if RUN_MODE == 'smoke' else MODELS
SEEDS_TO_RUN = (11,) if RUN_MODE == 'smoke' else SEEDS

META = ['row_id', 'FlightDate', 'scheduled_dep_utc', 'checkpoint_utc', 'wx_obs_time']
TARGET = 'LATE'
CAT_COLS = ['Origin', 'Dest', 'Reporting_Airline', 'wx_skyc1_T2h']
FOLDS = [
    (1, '2025-09-01', '2025-09-21'),
    (2, '2025-09-22', '2025-10-12'),
    (3, '2025-10-13', '2025-11-02'),
    (4, '2025-11-03', '2025-11-28'),
]

assert (DATA_DIR / 'train_T2h.csv').exists(), f'Missing training export in {DATA_DIR.resolve()}'
assert (DATA_DIR / 'validation_T2h.csv').exists(), f'Missing validation export in {DATA_DIR.resolve()}'
assert not (DATA_DIR / 'test_T2h.csv').exists() or True  # Explicitly never load the December file.
print(f'Using data: {DATA_DIR.resolve()}')
print(f'Writing handoff files: {OUT_DIR.resolve()}')

In [ ]:
def load_development_data(data_dir):
    train = pd.read_csv(data_dir / 'train_T2h.csv', low_memory=False)
    valid = pd.read_csv(data_dir / 'validation_T2h.csv', low_memory=False)
    for frame in (train, valid):
        frame['scheduled_dep_utc'] = pd.to_datetime(frame['scheduled_dep_utc'], utc=True)
        frame['checkpoint_utc'] = pd.to_datetime(frame['checkpoint_utc'], utc=True)
        assert frame['row_id'].is_unique, 'row_id must be unique within each export'
        assert set(META + [TARGET]).issubset(frame.columns)
    assert set(train.row_id).isdisjoint(valid.row_id), 'Development splits must not overlap'
    return train, valid

def feature_columns(frame):
    cols = [c for c in frame.columns if c not in META + [TARGET]]
    assert len(cols) == 35, f'Expected 35 raw predictors; found {len(cols)}'
    assert set(CAT_COLS).issubset(cols)
    return cols

def make_folds(train, valid):
    folds = []
    for fold_id, start, end in FOLDS:
        score = valid.loc[valid.FlightDate.between(start, end)].copy()
        first_checkpoint = score.checkpoint_utc.min()
        # Preserve Person 2's conservative 12-hour observability buffer.
        earlier = valid.loc[(valid.FlightDate < start) &
                            (valid.scheduled_dep_utc <= first_checkpoint - pd.Timedelta(hours=12))]
        fit = pd.concat([train, earlier], ignore_index=True).sort_values(['checkpoint_utc', 'row_id'])
        score = score.sort_values(['checkpoint_utc', 'row_id'])
        assert fit.checkpoint_utc.max() < score.checkpoint_utc.min()
        folds.append((fold_id, fit.reset_index(drop=True), score.reset_index(drop=True)))
    return folds

train_df, validation_df = load_development_data(DATA_DIR)
FEATURES = feature_columns(train_df)
print('train:', train_df.shape, '| validation:', validation_df.shape)
display(pd.DataFrame({
    'rows': [len(train_df), len(validation_df)],
    'late_rate': [train_df.LATE.mean(), validation_df.LATE.mean()],
    'start': [train_df.FlightDate.min(), validation_df.FlightDate.min()],
    'end': [train_df.FlightDate.max(), validation_df.FlightDate.max()],
}, index=['train', 'validation']))

In [ ]:
def chronological_early_stop_split(frame, fraction=0.10):
    """Return fit/early-stop blocks, with the newest 10% held out chronologically."""
    cut = max(1, int(len(frame) * (1 - fraction)))
    return frame.iloc[:cut].copy(), frame.iloc[cut:].copy()

def native_categories(fit, *others):
    """Fit categorical levels only on the historical training portion."""
    levels = {c: pd.Index(fit[c].astype('string').fillna('__MISSING__').unique()) for c in CAT_COLS}
    result = []
    for frame in (fit,) + others:
        x = frame[FEATURES].copy()
        for c in CAT_COLS:
            x[c] = pd.Categorical(x[c].astype('string').fillna('__MISSING__'), categories=levels[c])
        result.append(x)
    return result

class TopDestGrouper(BaseEstimator, TransformerMixin):
    """Fold-safe destination grouping so HGB native categoricals stay below max_bins."""
    def __init__(self, top_k=20):
        self.top_k = top_k
    def fit(self, X, y=None):
        self.top_dests_ = set(X['Dest'].astype('string').fillna('__MISSING__').value_counts().head(self.top_k).index)
        return self
    def transform(self, X):
        out = X.copy()
        dest = out['Dest'].astype('string').fillna('__MISSING__')
        out['Dest'] = dest.where(dest.isin(self.top_dests_), 'Other')
        return out

def hist_pipeline():
    numeric = [c for c in FEATURES if c not in CAT_COLS]
    # Ordinal coding is required because HistGradientBoosting has no native string support.
    prep = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)),
        ]), CAT_COLS),
    ], verbose_feature_names_out=False)
    return Pipeline([
        ('top_dest', TopDestGrouper(top_k=20)),
        ('prep', prep),
        # sklearn's classic GradientBoosting is impractical at this scale. HGB is the documented substitute.
        ('model', HistGradientBoostingClassifier(max_iter=350, learning_rate=0.06, max_leaf_nodes=31,
            l2_regularization=1.0, early_stopping=False, random_state=0,
            categorical_features=[False] * len(numeric) + [True] * len(CAT_COLS))),
    ])

def fit_predict(model_name, fit_block, early_block, score_block, seed):
    y_fit = fit_block[TARGET].astype(int)
    y_early = early_block[TARGET].astype(int)
    if model_name == 'hist_gradient_boosting':
        model = hist_pipeline()
        model.set_params(model__random_state=seed)
        model.fit(fit_block[FEATURES], y_fit)
        return model, model.predict_proba(score_block[FEATURES])[:, 1], 350

    X_fit, X_early, X_score = native_categories(fit_block, early_block, score_block)
    if model_name == 'xgboost':
        model = xgb.XGBClassifier(
            n_estimators=1500, learning_rate=0.035, max_depth=8, min_child_weight=20,
            subsample=0.85, colsample_bytree=0.85, objective='binary:logistic',
            eval_metric='aucpr', enable_categorical=True, tree_method='hist',
            early_stopping_rounds=100, n_jobs=N_JOBS, random_state=seed)
        model.fit(X_fit, y_fit, eval_set=[(X_early, y_early)], verbose=False)
        best = int(model.best_iteration + 1) if model.best_iteration is not None else 1500
        return model, model.predict_proba(X_score, iteration_range=(0, best))[:, 1], best
    if model_name == 'lightgbm':
        model = lgb.LGBMClassifier(
            objective='binary', n_estimators=1500, learning_rate=0.035, num_leaves=63,
            min_child_samples=100, subsample=0.85, colsample_bytree=0.85,
            reg_lambda=1.0, n_jobs=N_JOBS, random_state=seed, verbosity=-1)
        model.fit(X_fit, y_fit, categorical_feature=CAT_COLS,
                  eval_set=[(X_early, y_early)], eval_metric='average_precision',
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        best = int(model.best_iteration_ or 1500)
        return model, model.predict_proba(X_score, num_iteration=best)[:, 1], best
    if model_name == 'catboost':
        # CatBoost requires categorical inputs as strings rather than pandas category values.
        for x in (X_fit, X_early, X_score):
            for c in CAT_COLS:
                x[c] = x[c].astype('string').fillna('__MISSING__').astype(str)
        model = CatBoostClassifier(
            loss_function='Logloss', eval_metric='PRAUC', iterations=1500, learning_rate=0.035,
            depth=8, l2_leaf_reg=5.0, random_seed=seed, thread_count=N_JOBS, verbose=False,
            allow_writing_files=False, od_type='Iter', od_wait=100)
        model.fit(X_fit, y_fit, cat_features=CAT_COLS, eval_set=(X_early, y_early), verbose=False)
        best = int(model.get_best_iteration() + 1) if model.get_best_iteration() >= 0 else 1500
        return model, model.predict_proba(X_score)[:, 1], best
    raise ValueError(model_name)

In [ ]:
# Full benchmark. Do not interrupt it merely because a single fold is slow.
started = time.time()
fold_scores, iteration_rows, oof_parts = [], [], []
for fold_id, train_fold, score_fold in make_folds(train_df, validation_df):
    if MAX_TRAIN_ROWS and len(train_fold) > MAX_TRAIN_ROWS:
        train_fold = train_fold.iloc[-MAX_TRAIN_ROWS:].reset_index(drop=True)
    fit_block, early_block = chronological_early_stop_split(train_fold)
    print(f'Fold {fold_id}: fit={len(fit_block):,}, early-stop={len(early_block):,}, score={len(score_fold):,}')
    oof = score_fold[['row_id', 'FlightDate', TARGET]].copy()
    oof['fold'] = fold_id
    for model_name in MODELS_TO_RUN:
        seed_probs, seed_ap, seed_auc, seed_iterations = [], [], [], []
        for seed in SEEDS_TO_RUN:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                model, probs, best_iteration = fit_predict(model_name, fit_block, early_block, score_fold, seed)
            seed_probs.append(probs)
            seed_ap.append(average_precision_score(score_fold[TARGET], probs))
            seed_auc.append(roc_auc_score(score_fold[TARGET], probs))
            seed_iterations.append(best_iteration)
        oof[f'prob_{model_name}'] = np.mean(seed_probs, axis=0)
        fold_scores.append({'model': model_name, 'fold': fold_id, 'ap': np.mean(seed_ap),
                            'roc_auc': np.mean(seed_auc), 'n_features': len(FEATURES),
                            'n_train_fit': len(fit_block), 'n_early_stop': len(early_block)})
        iteration_rows.append({'model': model_name, 'fold': fold_id, 'seed_iterations': seed_iterations,
                               'mean_iteration': float(np.mean(seed_iterations))})
        print(f'  {model_name}: AP={np.mean(seed_ap):.5f}, ROC-AUC={np.mean(seed_auc):.5f}, iterations={seed_iterations}')
    oof_parts.append(oof)

fold_scores = pd.DataFrame(fold_scores)
summary = (fold_scores.groupby('model', as_index=False)
           .agg(ap_mean=('ap', 'mean'), ap_std=('ap', 'std'),
                roc_auc_mean=('roc_auc', 'mean'), roc_auc_std=('roc_auc', 'std'))
           .sort_values('ap_mean', ascending=False))
fold_scores.to_csv(OUT_DIR / 'person3_fold_scores.csv', index=False)
summary.to_csv(OUT_DIR / 'person3_summary_scores.csv', index=False)
pd.DataFrame(iteration_rows).to_csv(OUT_DIR / 'person3_early_stopping.csv', index=False)
oof = pd.concat(oof_parts, ignore_index=True)
oof.to_parquet(OUT_DIR / 'person3_oof_predictions.parquet', index=False)
settings = {
    'run_mode': RUN_MODE, 'models': MODELS_TO_RUN, 'seeds': list(SEEDS_TO_RUN),
    'folds': FOLDS, 'features': FEATURES, 'early_stopping': {
        'block': 'chronologically newest 10% of each fold training set',
        'rounds': 100,
        'hist_gradient_boosting': 'disabled; uses fixed 350 iterations because sklearn internal validation is not chronological',
    },
    'elapsed_seconds': round(time.time() - started, 2),
}
(OUT_DIR / 'person3_chosen_settings.json').write_text(json.dumps(settings, indent=2))
display(summary)
print(f'Finished in {settings["elapsed_seconds"] / 60:.1f} minutes. Files for Person 4 are in {OUT_DIR.resolve()}')

## Handoff and next step

Give Person 4 these five files: `person3_summary_scores.csv`, `person3_fold_scores.csv`, `person3_oof_predictions.parquet`, `person3_early_stopping.csv`, and `person3_chosen_settings.json`. Do not run `test_T2h.csv` here. Person 4 first selects the configuration from development results, performs the prescribed feature ablation, freezes all choices, and evaluates December once.